<a href="https://colab.research.google.com/github/kojeda603/analisis_exploratorio_de_datos/blob/main/11_datos_crudos_tareas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [110]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [111]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [112]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/ciencia_de_datos/datos_crudos_tarea.csv")

In [113]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        1020 non-null   int64  
 1   id_paciente       1020 non-null   int64  
 2   fecha_consulta    1020 non-null   object 
 3   edad              1006 non-null   object 
 4   sexo              917 non-null    object 
 5   peso_kg           1006 non-null   object 
 6   altura_m          1006 non-null   object 
 7   presion_arterial  1020 non-null   object 
 8   glucosa_mg_dL     923 non-null    object 
 9   colesterol_total  886 non-null    object 
 10  fuma              908 non-null    object 
 11  actividad_fisica  929 non-null    object 
 12  imc_calculado     0 non-null      float64
dtypes: float64(1), int64(2), object(10)
memory usage: 103.7+ KB


In [114]:
print(df.shape)
print(110* "*")
print(df.dtypes)
print(110* "*")
print(df.isnull().sum())
print(110* "*")
print(df.duplicated().sum())
print(110* "*")
df.head()

(1020, 13)
**************************************************************************************************************
Unnamed: 0            int64
id_paciente           int64
fecha_consulta       object
edad                 object
sexo                 object
peso_kg              object
altura_m             object
presion_arterial     object
glucosa_mg_dL        object
colesterol_total     object
fuma                 object
actividad_fisica     object
imc_calculado       float64
dtype: object
**************************************************************************************************************
Unnamed: 0             0
id_paciente            0
fecha_consulta         0
edad                  14
sexo                 103
peso_kg               14
altura_m              14
presion_arterial       0
glucosa_mg_dL         97
colesterol_total     134
fuma                 112
actividad_fisica      91
imc_calculado       1020
dtype: int64
***************************************************

,Unnamed: 0,id_paciente,fecha_consulta,edad,sexo,peso_kg,altura_m,presion_arterial,glucosa_mg_dL,colesterol_total,fuma,actividad_fisica,imc_calculado
0,0,524,2022-01-23 00:00:00,73,MUJER,80.6,2.38,89/97,NaN,-100,0,alta,NaN
1,1,603,2023-03-02 00:00:00,7,X,126.3,1.18,88/79,95,-100,0,Sedentario,NaN
2,2,527,2022-04-28 00:00:00,68,F,37.7,20,118/51,500,-100,no,Sedentario,NaN
3,3,32,2022-05-06 00:00:00,96,f,47.5,0.45,195/105,300,bueno,SI,Sedentario,NaN
4,4,617,2023-06-07 00:00:00,72,Mujer,198.8,0.81,126/87,85,120,0,Baja,NaN


#FECHA

In [115]:
# 'Unnamed: 0' es el índice antiguo guardado por error al exportar el CSV
df = df.drop(columns=['Unnamed: 0'])

# Quitamos filas duplicadas completas y duplicados de id_paciente (nos quedamos con el primero)
antes = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset='id_paciente', keep='first')
print(f"Filas eliminadas por duplicados: {antes - len(df)}")
print(f"Shape tras quitar duplicados: {df.shape}")

Filas eliminadas por duplicados: 20
Shape tras quitar duplicados: (1000, 12)


In [116]:
# errors='coerce' convierte las fechas inválidas ('2021/02/30', 'fecha inválida',
# '2022-02-29' que no es bisiesto, '2020-13-01', etc.) a NaT
df['fecha_consulta'] = pd.to_datetime(df['fecha_consulta'], errors='coerce')

print("Fechas inválidas convertidas a NaT:", df['fecha_consulta'].isna().sum())
print("Rango:", df['fecha_consulta'].min(), "→", df['fecha_consulta'].max())

antes = len(df)
df = df.dropna(subset=['fecha_consulta'])
print(f"\nFilas eliminadas por fecha inválida: {antes - len(df)}")
print(f"Shape tras eliminar: {df.shape}")

Fechas inválidas convertidas a NaT: 10
Rango: 2020-01-01 00:00:00 → 2023-12-30 00:00:00

Filas eliminadas por fecha inválida: 10
Shape tras eliminar: (990, 12)


#EDAD

In [117]:
# Valores NO numéricos que aparecen en la columna 'edad'
no_numericos = df[pd.to_numeric(df['edad'], errors='coerce').isna()]['edad'].dropna()

print("Valores únicos no numéricos en 'edad':", no_numericos.unique().tolist())
print("\nFrecuencia:")
print(no_numericos.value_counts())

Valores únicos no numéricos en 'edad': ['ochenta', 'desconocido', 'noventa']

Frecuencia:
edad
ochenta        14
desconocido    12
noventa        12
Name: count, dtype: int64


In [118]:
# Mapeo de palabras-números que aparecen en edad, peso y altura
palabras_num = {
    'ochenta': 80,
    'noventa': 90,
    'desconocido': np.nan
}

def limpiar_numerico(serie):
    """Convierte una columna mixta (texto + números) a float.
    - Reemplaza palabras por su equivalente numérico
    - Lo que no sea convertible queda como NaN
    """
    serie = serie.astype(str).str.strip().str.lower()
    serie = serie.replace(palabras_num)
    return pd.to_numeric(serie, errors='coerce')

In [119]:
df['edad'] = limpiar_numerico(df['edad'])

# Rango biológicamente: 0 a 120 años
# Todo lo que esté fuera (negativos, 150, 200, 999) lo marcamos como NaN
mask_edad_invalida = (df['edad'] < 0) | (df['edad'] > 120)
print(f"Edades fuera de rango → NaN: {mask_edad_invalida.sum()}")
df.loc[mask_edad_invalida, 'edad'] = np.nan

df['edad'].describe()

Edades fuera de rango → NaN: 100


,edad
count,865.000000
mean,54.523699
std,31.836129
min,0.000000
25%,27.000000
50%,54.000000
75%,82.000000
max,110.000000


In [120]:
antes = len(df)
df = df.dropna(subset=['fecha_consulta'])
print(f"\nFilas eliminadas por fecha inválida: {antes - len(df)}")
print(f"Shape tras eliminar: {df.shape}")


Filas eliminadas por fecha inválida: 0
Shape tras eliminar: (990, 12)


In [121]:
# Valores NO numéricos que aparecen en la columna 'edad'
no_numericos = df[pd.to_numeric(df['edad'], errors='coerce').isna()]['edad'].dropna()

print("Valores únicos no numéricos en 'edad':", no_numericos.unique().tolist())
print("\nFrecuencia:")
print(no_numericos.value_counts())

Valores únicos no numéricos en 'edad': []

Frecuencia:
Series([], Name: count, dtype: int64)


#SEXO

In [122]:
# Valores únicos en 'sexo' y su frecuencia
print("Valores únicos en 'sexo':", df['sexo'].unique().tolist())
print("\nFrecuencia:")
print(df['sexo'].value_counts(dropna=False))

Valores únicos en 'sexo': ['MUJER', 'X', 'F', 'f', 'Mujer', 'm', 'Hombre', 'M', nan, 'H']

Frecuencia:
sexo
M         111
f         107
NaN       103
Mujer     102
Hombre    100
m         100
MUJER      97
F          92
X          91
H          87
Name: count, dtype: int64


In [123]:
# Tenemos: F, f, MUJER, Mujer, M, m, H, Hombre, X, nan
# Estandarizamos a: 'F' (femenino), 'M' (masculino), NaN (desconocido/X)
mapa_sexo = {
    'f': 'F', 'mujer': 'F', 'MUJER': 'F',
    'm': 'M', 'h': 'M', 'hombre': 'M',
    'X': np.nan, 'nan': np.nan
}
df['sexo'] = df['sexo'].astype(str).str.strip().str.lower().map(mapa_sexo)
print(df['sexo'].value_counts(dropna=False))

sexo
F      398
M      398
NaN    194
Name: count, dtype: int64


PESO

In [124]:
# Valores únicos en 'peso' y su frecuencia
print("Valores únicos en 'peso_kg':", df['peso_kg'].unique().tolist())
print("\nFrecuencia:")
print(df['peso_kg'].value_counts(dropna=False))

Valores únicos en 'peso_kg': ['80.6', '126.3', '37.7', '47.5', '198.8', '104.7', '18.2', '92.7', '93.9', '73.4', '-54.8', '196.4', '193.1', '159.7', '5000', '24.4', '149.5', '148.3', '20.9', '105.7', '44.6', '98.0', '177.2', '183.2', '173.9', '129.9', '199.5', 'setenta', '26.6', '29.6', '116.5', '67.7', '42.9', '139.8', '165.6', '149.0', '57.2', '58.6', '62.8', '-99.0', '20.1', '140.5', 'noventa', '103.8', '188.2', '-47.2', '61.2', '11.8', '59.0', '25.1', '12.1', '-34.4', '-82.3', '187.9', '9999', '153.4', '69.4', '170.1', '124.9', '16.1', '161.5', '5.2', '68.8', '37.5', '80.0', '13.3', '5.6', '55.8', '192.4', '34.8', '86.7', '98.5', '-2.3', '115.6', '20.8', '86.9', '43.3', '14.6', '195.4', '159.9', '102.1', '123.7', '12.8', '-1.7', '176.1', '135.6', '44.0', '3.3', '57.9', '56.2', '57.6', '10.3', '197.5', '47.8', '54.4', '144.0', '80.8', '26.1', '124.4', '140.9', '116.2', '-48.7', '172.1', '164.7', '178.3', '90.3', '82.8', '89.7', '1000', '127.3', '186.6', '27.1', '152.6', '64.5', '119

In [125]:
df['peso_kg'] = limpiar_numerico(df['peso_kg'])

# Rango razonable en humanos: 2 kg (recién nacido) a 200 kg
# Valores como -54, 5000, 9999 → NaN
mask_peso = (df['peso_kg'] < 2) | (df['peso_kg'] > 200)
print(f"Pesos fuera de rango → NaN: {mask_peso.sum()}")
df.loc[mask_peso, 'peso_kg'] = np.nan

df['peso_kg'].describe()

Pesos fuera de rango → NaN: 128


,peso_kg
count,835.000000
mean,97.980599
std,56.454881
min,2.000000
25%,49.600000
50%,94.100000
75%,145.850000
max,199.500000


In [126]:
# Valores únicos en 'peso' y su frecuencia
print("\nFrecuencia:")
print(df['peso_kg'].value_counts(dropna=False))


Frecuencia:
peso_kg
NaN      155
90.0      16
80.0      12
12.8       4
134.0      3
        ... 
112.2      1
168.2      1
52.6       1
89.3       1
16.3       1
Name: count, Length: 661, dtype: int64


#ALTURA

In [127]:
# Valores únicos en 'sexo' y su frecuencia
print("Valores únicos en 'altura_m':", df['altura_m'].unique().tolist())
print("\nFrecuencia:")
print(df['altura_m'].value_counts(dropna=False))

Valores únicos en 'altura_m': ['2.38', '1.18', '20', '0.45', '0.81', '1.11', '2.17', '1.47', '0.82', '2.47', '1.78', '1.44', '0.73', '2.27', '0', '0.51', '2.3', '0.92', '2.48', '2.37', '1.79', 'desconocido', '1.67', '-0.67', '2.26', '-0.73', '2.29', '1.2', '2.36', '-0.9', '-0.5', '0.7', '1.24', '1.03', '0.76', '1.07', 'alto', '1.74', '0.67', '2.13', '1.76', '0.59', '-1.37', '1.19', '1.28', '1.77', '0.57', '2.28', '0.56', '1.45', '0.61', '1.31', '1.96', '2.03', '0.88', '2.46', '1.36', '2.49', '1.91', 'bajo', '1.61', '-1.76', '1.63', '-1.16', '1.82', '0.96', '2.33', '1.83', '-1.13', '0.87', '0.47', '1.29', '1.06', '10', '1.4', '1.88', '1.8', '2.22', nan, '1.49', '1.48', '1.5', '0.97', '1.37', '1.46', '0.89', '2.05', '1.15', '0.43', '1.1', '1.75', '1.97', '0.68', '1.98', '0.86', '2.44', '0.62', '0.8', '0.44', '1.94', '-2.28', '1.32', '0.41', '1.59', '2.2', '-2.0', '-1.26', '1.81', '0.84', '-0.63', '1.62', '0.71', '0.75', '1.13', '1.33', '-1.51', '2.34', '-0.46', '2.07', '0.42', '0.53', '0

In [128]:
df['altura_m'] = limpiar_numerico(df['altura_m'])

# Rango razonable: 0.4 m a 2.3 m
# Valores como 10, 20, 0, negativos → NaN
mask_alt = (df['altura_m'] < 0.4) | (df['altura_m'] > 2.3)
print(f"Alturas fuera de rango → NaN: {mask_alt.sum()}")
df.loc[mask_alt, 'altura_m'] = np.nan

df['altura_m'].describe()

Alturas fuera de rango → NaN: 210


,altura_m
count,718.000000
mean,1.356031
std,0.558318
min,0.400000
25%,0.870000
50%,1.340000
75%,1.850000
max,2.300000


In [129]:
# Valores únicos en 'sexo' y su frecuencia
print("\nFrecuencia:")
print(df['altura_m'].value_counts(dropna=False))


Frecuencia:
altura_m
NaN     272
1.15      9
1.67      8
2.16      8
2.07      8
       ... 
0.83      1
1.39      1
1.66      1
0.91      1
2.14      1
Name: count, Length: 190, dtype: int64


Glucosa y Colesterol

In [130]:
# valores negativos (-20,-100) y valores imposiblemente altos (500, 600)

df['glucosa_mg_dL']    = pd.to_numeric(df['glucosa_mg_dL'],    errors='coerce')
df['colesterol_total'] = pd.to_numeric(df['colesterol_total'], errors='coerce')

# Rangos clínicamente plausibles (incluso valores anormales reales)
# Glucosa:    40–400 mg/dL
# Colesterol: 80–400 mg/dL
df.loc[(df['glucosa_mg_dL']    < 40)  | (df['glucosa_mg_dL']    > 400), 'glucosa_mg_dL']    = np.nan
df.loc[(df['colesterol_total'] < 80)  | (df['colesterol_total'] > 400), 'colesterol_total'] = np.nan

df[['glucosa_mg_dL', 'colesterol_total']].describe()

,glucosa_mg_dL,colesterol_total
count,535.000000,350.000000
mean,142.588785,220.314286
std,87.646502,95.534470
min,50.000000,120.000000
25%,85.000000,120.000000
50%,110.000000,200.000000
75%,180.000000,350.000000
max,300.000000,350.000000


FUMA

In [131]:
# Valores únicos en 'fuma' y su frecuencia
print("Valores únicos en 'fuma':", df['fuma'].unique().tolist())
print("\nFrecuencia:")
print(df['fuma'].value_counts(dropna=False))

Valores únicos en 'fuma': ['0', 'no', 'SI', 'Fumador', 'NO', 'No', '1', 'Sí', nan, 'si']

Frecuencia:
fuma
0          109
no         109
NaN        109
No         103
SI         102
Sí         100
NO          99
Fumador     92
1           85
si          82
Name: count, dtype: int64


In [132]:
# Valores originales: 0, 1, 'Fumador', 'SI', 'Sí', 'si', 'NO', 'No', 'no', 'nan'
mapa_fuma = {
    '1': 1, 'si': 1, 'sí': 1, 'fumador': 1,
    '0': 0, 'no': 0,
    'nan': np.nan
}
df['fuma'] = df['fuma'].astype(str).str.strip().str.lower().map(mapa_fuma)
print(df['fuma'].value_counts(dropna=False))

fuma
1.0    461
0.0    420
NaN    109
Name: count, dtype: int64


actividad_fisica


In [133]:
# Valores únicos en 'actividad_fisica' y su frecuencia
print("Valores únicos en 'actividad_fisica':", df['actividad_fisica'].unique().tolist())
print("\nFrecuencia:")
print(df['actividad_fisica'].value_counts(dropna=False))

Valores únicos en 'actividad_fisica': ['alta', 'Sedentario', 'Baja', '0', 'baja', 'Alta', nan, '1', 'Media', '2', 'media']

Frecuencia:
actividad_fisica
Media         110
1              96
alta           96
2              96
Alta           93
Baja           89
NaN            88
0              86
baja           82
media          82
Sedentario     72
Name: count, dtype: int64


In [134]:
# Mezcla horrible: texto ('Sedentario','baja','media','alta' con mayúsculas varias)
# y números (0,1,2). Unificamos todo a la escala de texto:
#   0 = Sedentario, 1 = Media, 2 = Alta   (interpretación estándar)
mapa_actividad = {
    '0': 'Sedentario',
    'sedentario': 'Sedentario',
    'baja': 'Baja',
    '1': 'Media',
    'media': 'Media',
    '2': 'Alta',
    'alta': 'Alta',
    'nan': np.nan
}
df['actividad_fisica'] = (
    df['actividad_fisica'].astype(str).str.strip().str.lower().map(mapa_actividad)
)

# Lo convertimos a categórico ordenado (útil para análisis posteriores)
df['actividad_fisica'] = pd.Categorical(
    df['actividad_fisica'],
    categories=['Sedentario', 'Baja', 'Media', 'Alta'],
    ordered=True
)
print(df['actividad_fisica'].value_counts(dropna=False))

actividad_fisica
Media         288
Alta          285
Baja          171
Sedentario    158
NaN            88
Name: count, dtype: int64


In [135]:
# La columna imc_calculado venía vacía → la calculamos nosotros
# IMC = peso (kg) / altura (m)²
df['imc_calculado'] = df['peso_kg'] / (df['altura_m'] ** 2)

# Redondeamos a 2 decimales
df['imc_calculado'] = df['imc_calculado'].round(2)

df['imc_calculado'].describe()

,imc_calculado
count,619.000000
mean,107.673166
std,156.442393
min,0.390000
25%,24.560000
50%,48.300000
75%,121.125000
max,1052.150000


In [136]:
print("="*55)
print("RESUMEN DATASET LIMPIO")
print("="*55)
print(f"Filas: {len(df)}  |  Columnas: {df.shape[1]}")
print("\nNulos por columna:")
print(df.isna().sum())
print("\nTipos finales:")
print(df.dtypes)
print("\nDescribe numérico:")
print(df.describe().round(2))

# Guardamos
df.to_csv('datos_limpios.csv', index=False)
print("\n✓ Guardado como 'datos_limpios.csv'")


RESUMEN DATASET LIMPIO
Filas: 990  |  Columnas: 12

Nulos por columna:
id_paciente           0
fecha_consulta        0
edad                125
sexo                194
peso_kg             155
altura_m            272
presion_arterial      0
glucosa_mg_dL       455
colesterol_total    640
fuma                109
actividad_fisica     88
imc_calculado       371
dtype: int64

Tipos finales:
id_paciente                  int64
fecha_consulta      datetime64[ns]
edad                       float64
sexo                        object
peso_kg                    float64
altura_m                   float64
presion_arterial            object
glucosa_mg_dL              float64
colesterol_total           float64
fuma                       float64
actividad_fisica          category
imc_calculado              float64
dtype: object

Describe numérico:
       id_paciente                 fecha_consulta    edad  peso_kg  altura_m  \
count       990.00                            990  865.00   835.00    718.00  